In [1]:
!git clone https://github.com/keiranoluv/final_project
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'final_project'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 33 (delta 6), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 16.21 KiB | 922.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353017, done.
remote: Counting objects: 100% (1276/1276), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 353017 (delta 1189), reused 1099 (delta 1099), pack-reused 351741 (from 3)
Receiving objects: 100% (353017/353017), 1.87 GiB | 32.06 MiB/s, done.
Resolving deltas: 100% (279198/279198), done.


In [2]:
%cd /kaggle/working/PaddleOCR

!python -m pip install -q -r requirements.txt
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --no-deps

/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 877.4 kB/s eta 0:00:00


In [3]:
import csv
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2")
OUTPUT_ROOT = Path("/kaggle/working/mthv2_labels")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    src = DATASET_ROOT / f"{split}.tsv"
    dst = OUTPUT_ROOT / f"{split}.txt"

    count = 0

    with src.open("r", encoding="utf-8") as fin, \
         dst.open("w", encoding="utf-8") as fout:

        reader = csv.DictReader(fin, delimiter="\t")

        for row in reader:
            fout.write(f'{row["image_path"]}\t{row["text"]}\n')
            count += 1

    print(f"{split}: {count:,} samples -> {dst}")

train: 72,563 samples -> /kaggle/working/mthv2_labels/train.txt
val: 7,753 samples -> /kaggle/working/mthv2_labels/val.txt
test: 25,262 samples -> /kaggle/working/mthv2_labels/test.txt


In [4]:
TRAIN_CHARS = Path(
    "/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2/train_characters.txt"
)

PADDLE_DICT = Path(
    "/kaggle/working/PaddleOCR/ppocr/utils/dict/ppocrv5_dict.txt"
)

train_chars = {
    line.rstrip("\r\n")
    for line in TRAIN_CHARS.open("r", encoding="utf-8")
}

paddle_chars = {
    line.rstrip("\r\n")
    for line in PADDLE_DICT.open("r", encoding="utf-8")
}

# bỏ dòng rỗng nhưng GIỮ space nếu dataset thực sự có
train_chars.discard("")
paddle_chars.discard("")

missing = train_chars - paddle_chars
covered = train_chars & paddle_chars

print("=== Character coverage ===")
print(f"MTHv2 train unique chars : {len(train_chars):,}")
print(f"PP-OCRv5 dict chars      : {len(paddle_chars):,}")
print(f"Covered                  : {len(covered):,}")
print(f"Missing                  : {len(missing):,}")
print(f"Coverage                 : {len(covered) / len(train_chars) * 100:.4f}%")

=== Character coverage ===
MTHv2 train unique chars : 6,063
PP-OCRv5 dict chars      : 18,383
Covered                  : 5,277
Missing                  : 786
Coverage                 : 87.0361%


In [5]:
!mkdir -p pretrained

!wget -O pretrained/PP-OCRv5_server_rec_pretrained.pdparams \
  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams

--2026-08-08 19:53:31--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 214594738 (205M) [application/octet-stream]
Saving to: ‘pretrained/PP-OCRv5_server_rec_pretrained.pdparams’

pretrained/PP-OCRv5 100%[===================>] 204.65M  8.22MB/s    in 33s     

2026-08-08 19:54:05 (6.17 MB/s) - ‘pretrained/PP-OCRv5_server_rec_pretrained.pdparams’ saved [214594738/214594738]



In [6]:
!wc -l /kaggle/input/datasets/zephyrvn/vocabylary-expanded/ppocrv5_mthv2_expanded.txt

19169 /kaggle/input/datasets/zephyrvn/vocabylary-expanded/ppocrv5_mthv2_expanded.txt


## B2 - Expand Vocabulary

### Mục tiêu

Sau khi phân tích lỗi của B1, phần lớn lỗi còn lại liên quan đến các ký tự xuất hiện trong MTHv2 nhưng không tồn tại trong vocabulary gốc của PP-OCRv5.

Đặc biệt, nhiều ký tự bị **deletion** dù đã xuất hiện nhiều lần trong train set, ví dụ:

- `䖏`
- `𫝹`
- `𢙉`
- `丗`
- `隂`
- `㑹`
- `𨚗`

Điều này cho thấy model có thể học đặc trưng ảnh của các ký tự này, nhưng output layer không thể dự đoán chúng nếu chúng không tồn tại trong character dictionary.

---

### Vocabulary gốc

Dictionary gốc của PP-OCRv5:

    ppocrv5_dict.txt

Số entry:

    18,383

---

### Cách tạo vocabulary mở rộng

Vocabulary mới được tạo bằng cách:

1. Giữ nguyên toàn bộ `18,383` entry của vocabulary gốc.
2. Đọc toàn bộ ký tự xuất hiện trong `train.tsv`.
3. Tìm các ký tự có trong train set nhưng chưa tồn tại trong vocabulary gốc.
4. Append các ký tự thiếu vào cuối dictionary.
5. Không sử dụng `val` hoặc `test` để xây dựng vocabulary nhằm tránh data leakage.

Kết quả:

    Base vocabulary      : 18,383
    Missing train chars  :    786
    Expanded vocabulary  : 19,169

File vocabulary mới:

    configs/dicts/ppocrv5_mthv2_expanded.txt

Trên Kaggle:

    /kaggle/input/datasets/zephyrvn/vocabylary-expanded/ppocrv5_mthv2_expanded.txt

---

### Kiểm tra vocabulary

Vocabulary mở rộng đã được verify:

    Total entries : 19,169
    Unique entries: 19,169
    Blank lines   : 0
    Duplicates    : 0

Phần vocabulary gốc được giữ nguyên thứ tự, và `786` ký tự mới được append từ dòng `18,384` trở đi.

Toàn bộ ký tự xuất hiện trong train set sau đó đều có thể biểu diễn bằng dictionary mới.

---

### Ảnh hưởng đến kiến trúc model

Khi đổi vocabulary từ `18,383` sang `19,169`, kích thước các output layer thay đổi.

PP-OCRv5 sử dụng:

    CTC classes = vocabulary size + 2
    GTC classes = vocabulary size + 6

Do đó:

    Original:
    CTC = 18,385
    GTC = 18,389

    Expanded:
    CTC = 19,171
    GTC = 19,175

Khi load official pretrained checkpoint, 4 tensor phụ thuộc vocabulary sẽ bị shape mismatch:

    head.ctc_head.fc.weight
    head.ctc_head.fc.bias
    head.gtc_head.embedding.embedding.weight
    head.gtc_head.tgt_word_prj.weight

Các layer còn lại vẫn load pretrained weights bình thường.

Đây là hành vi expected của experiment B2-01.

---

## B2-01 Experiment

B2-01 sử dụng cùng training recipe với B1:

    Model       : PP-OCRv5_server_rec
    Init        : official pretrained
    Epochs      : 20
    GPUs        : 2
    Batch/GPU   : 64
    Eval step   : 500
    Train split : same as B1
    Val split   : same as B1

Khác biệt chính:

    B1   : original vocabulary (18,383)
    B2-01 : expanded vocabulary (19,169)

Mục tiêu của B2-01 là đánh giá mức cải thiện OCR khi mở rộng output vocabulary cho các ký tự đặc thù của MTHv2.

---

### Expected outcome

Kỳ vọng B2-01 sẽ giảm rõ rệt:

- deletion errors
- OOV-related errors
- lỗi trên các historical / rare Chinese characters

Các metric cần so sánh với B1:

    CER
    Exact Match Accuracy
    Substitution / Deletion / Insertion
    OOV-related deletions
    Recall trên 786 ký tự mới
    Inference speed
    VRAM usage

In [7]:
# %cd /kaggle/working/PaddleOCR

# !python -m paddle.distributed.launch \
#   --gpus "0,1" \
#   tools/train.py \
#   -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml \
#   -o \
#   Global.pretrained_model=./pretrained/PP-OCRv5_server_rec_pretrained.pdparams \
#   Global.epoch_num=20 \
#   Global.save_model_dir=/kaggle/working/final_project/outputs/B1_20epochs_2gpu_bs64 \
#   Global.eval_batch_step="[0,500]" \
#   Train.loader.batch_size_per_card=64 \
#   Train.sampler.first_bs=64 \
#   Train.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
#   Train.dataset.label_file_list='["/kaggle/working/mthv2_labels/train.txt"]' \
#   Eval.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
#   Eval.dataset.label_file_list='["/kaggle/working/mthv2_labels/val.txt"]'

In [8]:
%cd /kaggle/working/PaddleOCR

!python -m paddle.distributed.launch \
  --gpus "0,1" \
  tools/train.py \
  -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml \
  -o \
  Global.pretrained_model=./pretrained/PP-OCRv5_server_rec_pretrained.pdparams \
  Global.character_dict_path=/kaggle/input/datasets/zephyrvn/vocabylary-expanded/ppocrv5_mthv2_expanded.txt \
  Global.epoch_num=20 \
  Global.save_model_dir=/kaggle/working/final_project/outputs/B2C_20epochs_2gpu_bs64_expand_vocab \
  Global.eval_batch_step="[0,500]" \
  Train.loader.batch_size_per_card=64 \
  Train.sampler.first_bs=64 \
  Train.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Train.dataset.label_file_list='["/kaggle/working/mthv2_labels/train.txt"]' \
  Eval.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Eval.dataset.label_file_list='["/kaggle/working/mthv2_labels/val.txt"]'

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-08-08 19:54:11,017 -----------  Configuration  ----------------------
LAUNCH INFO 2026-08-08 19:54:11,017 auto_cluster_config: 0
LAUNCH INFO 2026-08-08 19:54:11,017 auto_parallel_config: None
LAUNCH INFO 2026-08-08 19:54:11,017 auto_tuner_json: None
LAUNCH INFO 2026-08-08 19:54:11,017 devices: 0,1
LAUNCH INFO 2026-08-08 19:54:11,017 elastic_level: -1
LAUNCH INFO 2026-08-08 19:54:11,017 elastic_timeout: 30
LAUNCH INFO 2026-08-08 19:54:11,017 enable_gpu_log: True
LAUNCH INFO 2026-08-08 19:54:11,017 gloo_port: 6767
LAUNCH INFO 2026-08-08 19:54:11,017 host: None
LAUNCH INFO 2026-08-08 19:54:11,017 ips: None
LAUNCH INFO 2026-08-08 